# DeepAR Deep Learning Forecasting

Trains a **DeepAR** probabilistic forecasting model (PyTorch Forecasting + Lightning)
on synthetic Walmart-style weekly sales. Self-contained — the setup cell generates the
source table, so it runs with no external data.

**Compute:** run on **Serverless GPU (AI Runtime)** — select an **A10** accelerator in the
Environment panel (right sidebar) before running. The `torch.cuda.is_available()` cell
below confirms the GPU is visible.

**Source table:** `shm_skunkworks_catalog.forecasting_brickfood.walmart_training`

In [ ]:
# Install DL deps first (GPU node has CUDA; torch wheel is CUDA-enabled)
%pip install --quiet torch pytorch-lightning pytorch-forecasting pytorch-tabular
dbutils.library.restartPython()

## Setup — self-contained data generation
Installs deps, then creates the source table (idempotent).

In [ ]:
# --- Setup: generate the source data (self-contained; idempotent) ---
spark.sql("CREATE SCHEMA IF NOT EXISTS shm_skunkworks_catalog.forecasting_brickfood")
spark.sql("""
CREATE TABLE IF NOT EXISTS shm_skunkworks_catalog.forecasting_brickfood.walmart_training AS
SELECT s.store AS store, d.dept AS dept,
       date_add(DATE'2022-01-07', 7*w.wk) AS `Date`,
       round(20000 + 6000*sin(2*pi()*(w.wk%52)/52.0) + 150*w.wk
             + s.store*1200 + d.dept*400 + (rand()*3000-1500), 2) AS `Weekly_Sales`
FROM (SELECT explode(sequence(0,149)) AS wk) w
CROSS JOIN (SELECT explode(array(1,2,3)) AS store) s
CROSS JOIN (SELECT explode(array(1,2,3)) AS dept) d
""")
print("walmart_training ready:",
      spark.table("shm_skunkworks_catalog.forecasting_brickfood.walmart_training").count(), "rows")

In [ ]:
import torch
print(torch.cuda.is_available())

## Load and prepare the series

Peek at the table, load a single store/dept series into pandas, add the `series`
group id and an integer `time_idx`, then build train/validation `TimeSeriesDataSet`s.
Approach adapted from the
[PyTorch Forecasting intro](https://medium.com/@mnitin3/pytorch-forecasting-introduction-to-time-series-forecasting-706cbc48768).

In [ ]:
%sql
select * from shm_skunkworks_catalog.forecasting_brickfood.walmart_training limit 10;

In [ ]:
df = spark.read.table("shm_skunkworks_catalog.forecasting_brickfood.walmart_training").filter("Store = 1 AND Dept = 1").toPandas()
df = df.drop(columns=["store", "dept"])
df["series"] = "dummy"
display(df)

In [ ]:
import pandas as pd

# PyTorch Forecasting needs an integer time_idx (one step per period per series).
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")
df["time_idx"] = ((df["Date"] - df["Date"].min()) / pd.Timedelta(weeks=1)).astype(int)
display(df.head())

In [ ]:
from pytorch_forecasting import TimeSeriesDataSet

max_encoder_length = 10
validation_cutoff = 30
max_prediction_length = 3

training_cutoff = df["time_idx"].max() - max_prediction_length - validation_cutoff

ts_kwargs = dict(
    time_idx="time_idx",
    target="Weekly_Sales",
    group_ids=["series"],
    min_encoder_length=5,           # can be less than max_encoder_length
    min_prediction_length=2,        # can be less than max_prediction_length
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_unknown_reals=["Weekly_Sales"],
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

training = TimeSeriesDataSet(df[df.time_idx <= training_cutoff], **ts_kwargs)
validation = TimeSeriesDataSet(
    df[(df.time_idx > training_cutoff)
       & (df.time_idx <= df["time_idx"].max() - max_prediction_length)],
    **ts_kwargs,
)
print(f"training_cutoff={training_cutoff}  train={len(training)}  val={len(validation)}")

In [ ]:
import warnings

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from pandas.errors import SettingWithCopyWarning
from pytorch_forecasting.metrics import NormalDistributionLoss
from pytorch_forecasting.models.deepar import DeepAR

warnings.simplefilter("error", category=SettingWithCopyWarning)

batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=5, verbose=False, mode="min"
)
lr_logger = LearningRateMonitor()

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices="auto",
    gradient_clip_val=0.1,
    limit_train_batches=30,
    limit_val_batches=3,
    callbacks=[lr_logger, early_stop_callback],
)

deepar = DeepAR.from_dataset(
    training,
    learning_rate=0.1,
    hidden_size=32,
    dropout=0.1,
    loss=NormalDistributionLoss(),
    log_interval=10,
    log_val_interval=3,
)
print(f"Number of parameters in network: {deepar.size() / 1e3:.1f}k")

trainer.fit(deepar, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

# Mean absolute error on the validation set. Keep actuals/predictions on the model's
# own device so this works on GPU or CPU without hardcoding "cuda".
device = deepar.device
actuals = torch.cat([y.to(device) for x, (y, weight) in iter(val_dataloader)])
predictions = deepar.predict(val_dataloader).to(device)
print(f"Validation MAE: {(actuals - predictions).abs().mean().item():.1f}")

# Plot a few forecasts (raw mode returns the full predictive distribution).
preds = deepar.predict(val_dataloader, mode="raw", return_x=True)
raw_predictions, x = preds.output, preds.x
try:
    for idx in range(3):
        deepar.plot_prediction(x, raw_predictions, idx=idx, add_loss_to_title=True)
except Exception as e:
    print("plot skipped:", e)

## Register to Unity Catalog
Log the trained DeepAR model to MLflow with a signature and register it to UC. UC
registration is wrapped in a try/except so the notebook still succeeds if the caller
lacks registry permissions (the model stays logged to the run either way).

In [ ]:
import mlflow, mlflow.pytorch
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")
UC_MODEL = "shm_skunkworks_catalog.forecasting_brickfood.deepar_walmart_forecast"

# UC requires a signature; build one from the val actuals/predictions computed above.
sig = infer_signature(actuals.detach().cpu().numpy(), predictions.detach().cpu().numpy())
with mlflow.start_run() as run:
    mlflow.pytorch.log_model(deepar, artifact_path="pytorch-model", signature=sig)
    try:
        result = mlflow.register_model(
            model_uri=f"runs:/{run.info.run_id}/pytorch-model", name=UC_MODEL)
        print(f"Registered {UC_MODEL} version {result.version}")
    except Exception as e:
        print("Model logged to run; UC registration skipped:", e)